# 02 — Data Quality Validation

Checks nulls, duplicates, keys, dates and referential integrity.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when
spark = SparkSession.builder.appName('RetailDataQuality').getOrCreate()
base='../Datasets/'
customers=spark.read.option('header',True).option('inferSchema',True).csv(base+'customers.csv')
orders=spark.read.option('header',True).option('inferSchema',True).csv(base+'orders_new.csv')


In [ ]:
def null_report(df):
    return df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])

null_report(customers).show()
null_report(orders).show()


In [ ]:
print('duplicate customer IDs:', customers.groupBy('customer_id').count().filter(col('count')>1).count())
print('duplicate order IDs:', orders.groupBy('order_id').count().filter(col('count')>1).count())


In [ ]:
orders = orders.withColumn('order_date', col('order_date').cast('timestamp'))
print('null order dates:', orders.filter(col('order_date').isNull()).count())


In [ ]:
orphans = orders.join(customers.select('customer_id'), 'customer_id', 'left_anti').count()
print('orphan orders:', orphans)
